# VoltesFive | HarborCart Phase 1 Data Preparation and Exploratory Analysis

Group: VoltesFive | Section: BSCS-3B | Instructor: Simon Gerard Granil

This Colab notebook analyzes the **UCI Online Retail** dataset as a proxy for fictional HarborCart. It does not contain actual HarborCart data. Source: [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail), Chen (2015), https://doi.org/10.24432/C5BW33 (CC BY 4.0). Historical UK sales are in GBP and do not represent Philippine customers.

**Instructions:** Download `Online Retail.xlsx` from UCI, open this notebook in Google Colab, run each cell in order, and upload that file when prompted. Keep the original dataset unchanged. Before submission, use the printed results and charts in your report and verify that your instructor approves the dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import zipfile

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 150)
OUTPUT = Path('harborcart_outputs')
OUTPUT.mkdir(exist_ok=True)
print('Libraries ready.')

## 1. Upload and inspect the dataset
Upload the original `.xlsx` file. For local testing, set `FILE_PATH` to the file path before running the cell.

In [ ]:
FILE_PATH = None  # Optional: e.g. '/content/Online Retail.xlsx'
if FILE_PATH is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError('Set FILE_PATH when running outside Google Colab.') from exc
    uploaded = files.upload()
    choices = [name for name in uploaded if name.lower().endswith('.xlsx')]
    if len(choices) != 1:
        raise ValueError('Upload exactly one .xlsx dataset file.')
    FILE_PATH = choices[0]
raw = pd.read_excel(FILE_PATH, dtype={'InvoiceNo': 'string', 'StockCode': 'string', 'CustomerID': 'string'})
raw.columns = raw.columns.str.strip()
required = ['InvoiceNo','StockCode','Description','Quantity','InvoiceDate','UnitPrice','CustomerID','Country']
missing_cols = sorted(set(required) - set(raw.columns))
if missing_cols:
    raise ValueError(f'Missing required dataset columns: {missing_cols}')
print(f'Original dimensions: {raw.shape[0]:,} rows × {raw.shape[1]} columns')
display(raw.head())
raw.info()

## 2. Data quality review
Exact duplicates are counted but retained by default: identical lines can reflect legitimate transactions. Review before making a removal decision.

In [ ]:
print('Missing values by column:')
display(raw.isna().sum().rename('missing_count').to_frame())
print(f'Exact duplicate rows: {raw.duplicated().sum():,}')
display(raw.describe(include='all').T)
raw.isna().sum().rename('missing_count').to_csv(OUTPUT / 'missing_values.csv')

## 3. Normalize fields and classify records
A line may meet multiple exclusion conditions. Independent counts therefore overlap; the sequential audit reports how many rows are removed at each stage. Cancellations are separated before applying the remaining purchase filters. No customer IDs are imputed.

In [ ]:
df = raw.copy()
for col in ['InvoiceNo','StockCode','CustomerID','Description','Country']:
    df[col] = df[col].astype('string').str.strip().replace('', pd.NA)
conversion_audit = []
for col in ['Quantity','UnitPrice']:
    before = df[col]
    converted = pd.to_numeric(before, errors='coerce')
    conversion_audit.append({'field': col, 'nonmissing_invalid_conversions': int((before.notna() & converted.isna()).sum())})
    df[col] = converted
before_date = df['InvoiceDate']
df['InvoiceDate'] = pd.to_datetime(before_date, errors='coerce')
conversion_audit.append({'field':'InvoiceDate', 'nonmissing_invalid_conversions':int((before_date.notna() & df['InvoiceDate'].isna()).sum())})
conversion_audit = pd.DataFrame(conversion_audit)
print('Invalid conversions (excluding originally missing values):')
display(conversion_audit)
conversion_audit.to_csv(OUTPUT / 'conversion_audit.csv', index=False)
df['IsCancelled'] = df['InvoiceNo'].str.upper().str.startswith('C', na=False).astype(bool)
df['InvoiceYear'] = df['InvoiceDate'].dt.year.astype('Int64')
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M').astype('string')

flags = pd.DataFrame({
    'cancelled_invoice': df['IsCancelled'],
    'missing_customer_id': df['CustomerID'].isna(),
    'missing_invoice_number': df['InvoiceNo'].isna(),
    'invalid_invoice_date': df['InvoiceDate'].isna(),
    'missing_or_nonpositive_quantity': df['Quantity'].isna() | df['Quantity'].le(0),
    'missing_or_nonpositive_unit_price': df['UnitPrice'].isna() | df['UnitPrice'].le(0),
})
print('Independent condition counts (may overlap):')
display(flags.sum().rename('rows').to_frame())
flags.sum().rename('rows').to_csv(OUTPUT / 'quality_condition_counts.csv')
cancellations = df.loc[flags['cancelled_invoice']].copy()
cancellations.to_csv(OUTPUT / 'cancelled_invoice_lines.csv', index=False)
print('Exact duplicates retained. Cancellations saved separately:', len(cancellations))

## 4. Sequential filtering and sales calculation
The resulting file is intended for **customer-level RFM**, not a complete accounting of all sales. Transactions without customer IDs are excluded; returns and cancellations are not netted against purchases.

In [ ]:
keep = pd.Series(True, index=df.index)
audit = []
for name in flags.columns:
    removed = keep & flags[name]
    keep &= ~flags[name]
    audit.append({'rule': name, 'removed_at_step': int(removed.sum()), 'remaining': int(keep.sum())})
clean = df.loc[keep].copy()
if clean.empty:
    raise ValueError('No eligible rows remain; check the uploaded file and audit.')
clean['TotalSales'] = clean['Quantity'] * clean['UnitPrice']
audit_df = pd.DataFrame(audit)
print('Sequential filtering audit:')
display(audit_df)
print(f'Eligible rows: {len(clean):,}; customers: {clean.CustomerID.nunique():,}')
audit_df.to_csv(OUTPUT / 'filter_audit.csv', index=False)
clean.to_csv(OUTPUT / 'rfm_eligible_transactions.csv', index=False)

## 5. Build customer RFM features
The reference date is the day after the latest eligible invoice. Frequency counts distinct invoice numbers, not product lines. Monetary value sums eligible purchase lines in GBP, without subtracting returns.

Additional report features: `AverageOrderValue` = Monetary / Frequency; `CustomerTenure` = days from first purchase to the reference date; and `UniqueProducts` = distinct non-missing StockCode values. These describe observed history and eligible purchases only.

In [ ]:
reference_date = clean['InvoiceDate'].max().normalize() + pd.Timedelta(days=1)
rfm = clean.groupby('CustomerID', as_index=False).agg(
    FirstPurchase=('InvoiceDate','min'),
    LastPurchase=('InvoiceDate','max'),
    UniqueProducts=('StockCode','nunique'),
    Frequency=('InvoiceNo','nunique'),
    Monetary=('TotalSales','sum')
)
rfm['Recency'] = (reference_date - rfm['LastPurchase'].dt.normalize()).dt.days
rfm['AverageOrderValue'] = rfm['Monetary'] / rfm['Frequency']
rfm['CustomerTenure'] = (reference_date - rfm['FirstPurchase'].dt.normalize()).dt.days
rfm = rfm[['CustomerID','FirstPurchase','LastPurchase','Recency','Frequency','Monetary','AverageOrderValue','CustomerTenure','UniqueProducts']]
assert (rfm['Recency'] >= 0).all() and (rfm['Frequency'] > 0).all()
print('RFM reference date:', reference_date.date())
display(rfm.head())
display(rfm[['Recency','Frequency','Monetary','AverageOrderValue','CustomerTenure','UniqueProducts']].describe())
rfm.to_csv(OUTPUT / 'customer_rfm.csv', index=False)
pd.DataFrame({'reference_date':[reference_date.date().isoformat()], 'eligible_rows':[len(clean)], 'customers':[len(rfm)], 'currency':['GBP']}).to_csv(OUTPUT / 'analysis_metadata.csv', index=False)

## 6. Exploratory charts
Charts below use only eligible customer-linked positive purchase lines. They are not net revenue and do not cover all transactions. Product quantities and monthly sales reflect this filtered subset.

In [ ]:
monthly = clean.groupby(clean['InvoiceDate'].dt.to_period('M'))['TotalSales'].sum()
monthly.index = monthly.index.to_timestamp()
top = clean.groupby('Description', dropna=True)['Quantity'].sum().nlargest(10).sort_values()
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
monthly.plot(ax=axes[0,0], marker='o', title='Monthly eligible purchase value'); axes[0,0].set_ylabel('GBP')
top.plot.barh(ax=axes[0,1], title='Top 10 products by units (eligible lines)'); axes[0,1].set_xlabel('Units')
axes[1,0].hist(rfm['Monetary'], bins=50); axes[1,0].set_title('Customer monetary value'); axes[1,0].set_xlabel('GBP')
axes[1,1].hist(rfm['Frequency'], bins=50); axes[1,1].set_title('Customer purchase frequency'); axes[1,1].set_xlabel('Distinct invoices')
fig.tight_layout(); fig.savefig(OUTPUT / 'eda_charts.png', dpi=180, bbox_inches='tight'); plt.show()
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, col in zip(axes,['Recency','Frequency','Monetary']):
    ax.boxplot(rfm[col].dropna(), vert=True); ax.set_title(col)
fig.suptitle('RFM distributions: investigate extreme values')
fig.tight_layout(); fig.savefig(OUTPUT / 'rfm_boxplots.png', dpi=180, bbox_inches='tight'); plt.show()

## 7. Export and download results
Review `filter_audit.csv`, the charts, and RFM summary before describing the results in your report. This notebook prepares data but does not train or evaluate a segmentation model.

In [ ]:
archive = Path('harborcart_phase1_outputs.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for file in sorted(OUTPUT.iterdir()):
        if file.is_file():
            z.write(file, arcname=file.name)
print('Saved:', archive.resolve())
print('Files:', ', '.join(p.name for p in sorted(OUTPUT.iterdir()) if p.is_file()))
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print('Download is available when running in Google Colab.')